# Banking Intent Assistant on BANKING77

PCAM ZG516 Applied Generative AI, Group Major Project, Assignment 1.

This notebook walks through the full workflow in order: dataset, model, baseline (E1), prompt comparison (E2),
decoding comparison (E3), structured output (E4), adaptation by retrieval (Req 8), failure analysis (E6), and the
final tool-assisted assistant (E7).

Every experiment lives in a `.py` file in the repository. The notebook imports those files; it does not copy the code.
Result tables are loaded from the saved `results_*.csv` files so the notebook runs in about one minute.
Each experiment also has an optional **rerun** cell that calls the model again on the 100-row subset (2 to 8 minutes each).

Full explanation of every step: `docs/PROGRESS.md`. Failure analysis: `docs/FAILURES.md`.

## 0. Setup

On Google Colab this clones the repository and installs the dependencies. Locally (after `uv sync`) it does nothing.

In [1]:
import os, sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/2025paml556-bhoomika/banking-agent.git
    %cd banking-agent
    %pip install -q transformers torch jsonschema pandas scikit-learn
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # run from the project root so imports and evaluation/ paths work

# One variable selects the model for every cell below. Default: Qwen/Qwen2.5-1.5B-Instruct.
# On a Colab GPU, uncomment the next line to run the larger model for comparison.
# os.environ["MODEL"] = "Qwen/Qwen2.5-3B-Instruct"

import pandas as pd
pd.set_option("display.max_colwidth", 80)

## 1. Dataset (Req 2)

BANKING77: 13,083 online-banking customer messages, each labelled with one of 77 fine-grained intents.
Two fields: `text` and `category`. The dataset has a fixed train/test split.

We use the **train split** (10,003 rows) only as a source of labelled examples for few-shot prompts and retrieval.
We evaluate on a **fixed 100-row sample of the test split** (`eval_subset.csv`, seed 0) so that every experiment
sees the same inputs and results are comparable.

In [2]:
train, test = pd.read_csv("data/train.csv"), pd.read_csv("data/test.csv")
subset = pd.read_csv("evaluation/eval_subset.csv")

print(f"train: {len(train)} rows, test: {len(test)} rows, intents: {train.category.nunique()}")
print(f"eval subset: {len(subset)} rows covering {subset.category.nunique()} intents")
print(f"message length (words): mean {train.text.str.split().str.len().mean():.1f}, max {train.text.str.split().str.len().max()}")
subset.head(8)

train: 10003 rows, test: 3080 rows, intents: 77
eval subset: 100 rows covering 53 intents
message length (words): mean 11.9, max 79


,text,category
0,How do I track my card?,card_arrival
1,How long should my new card take to arrive?,card_arrival
2,What foreign exchange rate will you use?,exchange_rate
3,when does the rate get determined,exchange_rate
4,Why am I unable to take money from my card but the statement says the transa...,pending_cash_withdrawal
5,I need my card quickly,card_delivery_estimate
6,When can I expect to receive my new card?,card_delivery_estimate
7,I can't get my card to work.,card_not_working


In [3]:
# Intent distribution in the train split (top and bottom 5) and one quirk: a single label with a capital letter.
counts = train.category.value_counts()
display(pd.concat([counts.head(5), counts.tail(5)]).to_frame("train rows"))
print("labels with a capital letter:", [c for c in counts.index if c != c.lower()])

,train rows
category,
card_payment_fee_charged,187
direct_debit_payment_not_recognised,182
balance_not_updated_after_cheque_or_cash_deposit,181
wrong_amount_of_cash_received,180
cash_withdrawal_charge,177
lost_or_stolen_card,82
card_swallowed,61
card_acceptance,59
virtual_card_not_working,41


labels with a capital letter: ['Refund_not_showing_up']


## 2. Model (Req 4, Req 5)

`llm.py` loads a pretrained Hugging Face causal language model, `Qwen/Qwen2.5-1.5B-Instruct`, and exposes one
function, `generate(messages, max_new_tokens, tools, **decoding)`. Every experiment and the final assistant call
this function. Nothing else touches the model.

The cell below shows the four stages of autoregressive generation on a project example:
tokenization, embeddings, the language-modelling head (next-token logits), and the generation loop.

In [4]:
import torch
from src.llm import tokenizer, model, generate, DEVICE, MODEL

message = "How do I track my card?"
ids = tokenizer(message)["input_ids"]
print("1. tokenization:", [tokenizer.decode([i]) for i in ids], "->", ids)

emb = model.get_input_embeddings()(torch.tensor([ids]).to(DEVICE))
print("2. embeddings:  ", tuple(emb.shape), "(1 sequence, tokens, hidden size)")

with torch.no_grad():
    logits = model(torch.tensor([ids]).to(DEVICE)).logits[0, -1]
top = torch.topk(logits.float().softmax(-1), 5)
print("3. LM head, top-5 next tokens after the message:", [(tokenizer.decode([i]), round(p.item(), 3)) for p, i in zip(top.values, top.indices)])

print("4. autoregressive generation (greedy, chat format):")
print("  ", generate([{"role": "user", "content": message}], max_new_tokens=40))

/Users/anujkumar/wilp_AIML/banking_agent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


16:27:45 llm: loading Qwen/Qwen2.5-1.5B-Instruct on mps


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 8276.41it/s]

16:28:01 llm: model ready, dtype torch.bfloat16, 1.5B parameters


...

2. embeddings:   (1, 7, 1536) (1 sequence, tokens, hidden size)


16:28:03 llm: LLM call: 36 prompt tokens, decoding={'do_sample': False}


3. LM head, top-5 next tokens after the message: [(' I', 0.097), (' To', 0.097), (' How', 0.075), (' You', 0.059), (' A', 0.031)]
4. autoregressive generation (greedy, chat format):


16:28:06 llm: LLM output (40 new tokens): "I'm sorry, but as an AI language model, I don't have access to your personal information or financial details. It's important to keep your sensitive data secure and not share it with anyone without"


   I'm sorry, but as an AI language model, I don't have access to your personal information or financial details. It's important to keep your sensitive data secure and not share it with anyone without


## 3. E1 Baseline (Req 5)

One simple prompt: the task, the 77 intent names, the message. No role, no examples, no output rules. Greedy decoding.
The output is free text, so `parse_intent` finds the longest intent name inside it.

**Result: accuracy 0.41.** For reference, a TF-IDF + logistic-regression classifier scores 0.79 on the same rows.

In [5]:
from src.baseline import build_prompt, parse_intent, INTENTS

print(build_prompt("How do I track my card?")[0]["content"][:200], "...\n")
r = pd.read_csv("evaluation/results/results_baseline.csv")
print(f"accuracy {r.correct.mean():.2f}, no valid intent {r.predicted.isna().sum()}, outputs with preamble {r.output.str.match(r'^\s*The').sum()}")
r[["text", "category", "output", "predicted", "correct"]].head(5)

16:28:07 baseline: loaded 77 intent names from the BANKING77 test split


Classify the customer message into one of these intents:
Refund_not_showing_up, activate_my_card, age_limit, apple_pay_or_google_pay, atm_support, automatic_top_up, balance_not_updated_after_bank_tran ...

accuracy 0.41, no valid intent 4, outputs with preamble 57


,text,category,output,predicted,correct
0,How do I track my card?,card_arrival,"The intent for the message ""How do I track my card?"" is:\n\nCard_delivery_es...",card_delivery_estimate,False
1,How long should my new card take to arrive?,card_arrival,Card_arrival,card_arrival,True
2,What foreign exchange rate will you use?,exchange_rate,"The intent for the message ""What foreign exchange rate will you use?"" is:\n\...",exchange_rate,True
3,when does the rate get determined,exchange_rate,"The intent for the message ""when does the rate get determined"" is not explic...",NaN,False
4,Why am I unable to take money from my card but the statement says the transa...,pending_cash_withdrawal,"The intent for this message is ""pending_transfer"".",pending_transfer,False


Optional rerun (about 2 minutes):

In [6]:
# %run -m src.baseline

## 4. E2 Prompt comparison (Req 6, Req 11)

`prompts.build_prompt(message, style)` is the reusable, parameterised template. Five styles, each adding one thing to the baseline:
role line, constraint line ("exact intent name, nothing else, always choose one"), 5 random solved examples, and examples + constraint.

**Result:** accuracy barely moves (0.40 to 0.43). Format changes a lot: the constraint removed all preamble text and halved the invented names.
Instructions cannot supply knowledge the model lacks, namely where the boundary between near-synonym labels lies.

In [7]:
from src.prompts import STYLES, build_prompt as prompt_by_style

rows = []
for s in STYLES:
    r = pd.read_csv(f"evaluation/results/results_{s}.csv")
    rows.append({"prompt": s, "accuracy": r.correct.mean(), "no_intent": r.predicted.isna().sum(), "preamble": r.output.str.match(r"^\s*The").sum()})
display(pd.DataFrame(rows))

print("--- constraint prompt (tail) ---")
print(prompt_by_style("How do I track my card?", "constraint")[0]["content"][-250:])

,prompt,accuracy,no_intent,preamble
0,zero_shot,0.41,4,57
1,role,0.40,4,30
2,constraint,0.40,2,0
3,few_shot,0.43,12,1
4,few_shot_constraint,0.43,8,0


--- constraint prompt (tail) ---
sa_or_mastercard, why_verify_identity, wrong_amount_of_cash_received, wrong_exchange_rate_for_cash_withdrawal
Reply with the exact intent name from the list and nothing else. Always choose exactly one intent.

Message: How do I track my card?
Intent:


Optional rerun (about 8 minutes):

In [8]:
# %run -m src.prompts

## 5. E3 Decoding comparison (Req 7)

Same prompt (`few_shot_constraint`), five decoding settings: greedy, temperature 0.7, temperature 1.5, top-k 50, top-p 0.9. Sampling runs are seeded.

**Result:** no setting beats greedy in a meaningful way (0.38 to 0.44). A one-label answer has one correct token sequence; randomness can only move away from the model's best guess.
Higher temperature produced more invented labels. Greedy is used for the intent step in the final system; sampling is used only for the free-text reply.

In [9]:
from src.decoding import SETTINGS

g = pd.read_csv("evaluation/results/results_decoding_greedy.csv")
rows = []
for name in SETTINGS:
    r = pd.read_csv(f"evaluation/results/results_decoding_{name}.csv")
    rows.append({"decoding": name, "params": SETTINGS[name], "accuracy": r.correct.mean(), "no_intent": r.predicted.isna().sum(), "rows flipped vs greedy": (r.correct != g.correct).sum()})
pd.DataFrame(rows)

,decoding,params,accuracy,no_intent,rows flipped vs greedy
0,greedy,{'do_sample': False},0.43,8,0
1,temperature_0.7,"{'do_sample': True, 'temperature': 0.7}",0.44,10,5
2,temperature_1.5,"{'do_sample': True, 'temperature': 1.5}",0.40,7,13
3,top_k_50,"{'do_sample': True, 'temperature': 1.0, 'top_k': 50}",0.44,7,7
4,top_p_0.9,"{'do_sample': True, 'temperature': 1.0, 'top_k': 0, 'top_p': 0.9}",0.38,11,11


Optional rerun (about 8 minutes):

In [10]:
# %run -m src.decoding

## 6. E4 Structured output (Req 9)

The model is asked for a JSON object `{"intent": ..., "confidence": ...}`. Every output is validated with `jsonschema` against a schema
whose `intent` field is an `enum` of the 77 names. The enum turns "is this a real intent?" into a hard check.

**Result:** 100/100 valid JSON, 86/100 pass the schema. All 14 rejections were invented intent names.
The `confidence` field was not informative: the model copied the example value (0.9) and gave the same mean confidence for right and wrong answers.

In [11]:
import json
from src.structured import SCHEMA, parse_json

print(json.dumps({**SCHEMA, "properties": {**SCHEMA["properties"], "intent": {"type": "string", "enum": "<77 intent names>"}}}, indent=2))

r = pd.read_csv("evaluation/results/results_structured.csv")
valid = r.error.isna()
print(f"\nvalid JSON 100/100, pass schema {valid.sum()}/100, accuracy all {r.correct.mean():.2f}, accuracy valid rows {r[valid].correct.mean():.2f}")
print("confidence values:", r.confidence.value_counts().to_dict())
print("\nrejected by schema:")
display(r[~valid][["text", "category", "output", "error"]].head(5))
print("validated samples:")
r[valid][["text", "category", "output", "correct"]].head(3)

{
  "type": "object",
  "properties": {
    "intent": {
      "type": "string",
      "enum": "<77 intent names>"
    },
    "confidence": {
      "type": "number",
      "minimum": 0,
      "maximum": 1
    }
  },
  "required": [
    "intent",
    "confidence"
  ],
  "additionalProperties": false
}

valid JSON 100/100, pass schema 86/100, accuracy all 0.43, accuracy valid rows 0.50
confidence values: {0.9: 84, 0.95: 16}

rejected by schema:


,text,category,output,error
10,How do I exchange currency?,exchange_via_app,"```json\n{\n ""intent"": ""exchange_currency"",\n ""confidence"": 0.9\n}\n```","schema: 'exchange_currency' is not one of ['Refund_not_showing_up', 'activat..."
22,help me obtain a virtual card,getting_virtual_card,"{\n ""intent"": ""get_virtual_card"",\n ""confidence"": 0.95\n}","schema: 'get_virtual_card' is not one of ['Refund_not_showing_up', 'activate..."
23,Where is the virtual card located?,getting_virtual_card,"{\n ""intent"": ""get_virtual_card"",\n ""confidence"": 0.9\n}","schema: 'get_virtual_card' is not one of ['Refund_not_showing_up', 'activate..."
24,Can I order a virtual card?,getting_virtual_card,"```json\n{\n ""intent"": ""get_virtual_card"",\n ""confidence"": 0.95\n}\n```","schema: 'get_virtual_card' is not one of ['Refund_not_showing_up', 'activate..."
25,Where are the virtual cards located?,getting_virtual_card,"{\n ""intent"": ""get_virtual_card"",\n ""confidence"": 0.9\n}","schema: 'get_virtual_card' is not one of ['Refund_not_showing_up', 'activate..."


validated samples:


,text,category,output,correct
0,How do I track my card?,card_arrival,"{\n ""intent"": ""card_delivery_estimate"",\n ""confidence"": 0.9\n}",False
1,How long should my new card take to arrive?,card_arrival,"{\n ""intent"": ""card_arrival"",\n ""confidence"": 0.9\n}",True
2,What foreign exchange rate will you use?,exchange_rate,"```json\n{\n ""intent"": ""exchange_rate"",\n ""confidence"": 0.9\n}\n```",True


Optional rerun (about 3 minutes):

In [12]:
# %run -m src.structured

## 7. Adaptation decision: retrieval few-shot (Req 8)

Prompting (E2) and decoding (E3) left accuracy at about 0.43. The failures were near-synonym confusions, which is missing knowledge, not missing instructions.
Three options: more prompting, retrieval (RAG), or fine-tuning (PEFT). We tested retrieval: for each message, the 5 most similar
train messages (TF-IDF cosine) become the few-shot examples.

**Result: accuracy 0.84**, up from 0.43. It beats a 5-NN vote on the same examples (0.77) and the TF-IDF classifier (0.79).
Fine-tuning was not needed to pass the classical baseline. Decision: RAG.

In [13]:
from src.retrieval import retrieve, build_prompt as retrieval_prompt

r = pd.read_csv("evaluation/results/results_retrieval.csv")
print(f"LLM + retrieved examples: {r.correct.mean():.2f}    5-NN vote only: {r.knn_correct.mean():.2f}    no valid intent: {r.predicted.isna().sum()}")
print(f"LLM right where vote wrong: {(r.correct & ~r.knn_correct).sum()}, LLM wrong where vote right: {(~r.correct & r.knn_correct).sum()}")

print("\nretrieved examples for 'How do I track my card?':")
display(retrieve("How do I track my card?")[["text", "category"]])

pd.DataFrame({"system": ["LLM, 5 random examples (E2)", "5-NN vote, no LLM", "TF-IDF + LogReg", "LLM, 5 retrieved examples"],
              "accuracy": [0.43, round(r.knn_correct.mean(), 2), 0.79, round(r.correct.mean(), 2)]})

LLM + retrieved examples: 0.84    5-NN vote only: 0.77    no valid intent: 3
LLM right where vote wrong: 12, LLM wrong where vote right: 5

retrieved examples for 'How do I track my card?':


,text,category
6544,how do i track the moeney,verify_source_of_funds
82,How can I track my card's delivery?,card_arrival
26,How do I track the card you sent me?,card_arrival
89,Is there a way to track my card?,card_arrival
95,Can you help me track my card?,card_arrival


,system,accuracy
0,"LLM, 5 random examples (E2)",0.43
1,"5-NN vote, no LLM",0.77
2,TF-IDF + LogReg,0.79
3,"LLM, 5 retrieved examples",0.84


Optional rerun (about 2 minutes):

In [14]:
# %run -m src.retrieval

## 8. E6 Evaluation and failure analysis (Req 12)

Criteria: accuracy, valid output, format discipline, reproducibility. The full table of 13 systems, 7 failure types and 11 worked cases are in `docs/FAILURES.md`.
Below: the summary across the four main systems and the rows that every system got wrong.

In [15]:
systems = {"zero_shot": "evaluation/results/results_zero_shot.csv", "few_shot_constraint": "evaluation/results/results_few_shot_constraint.csv",
           "structured": "evaluation/results/results_structured.csv", "retrieval": "evaluation/results/results_retrieval.csv"}
cmp = subset.copy()
for k, f in systems.items():
    cmp[k] = pd.read_csv(f).correct
cmp["n_right"] = cmp[list(systems)].sum(axis=1)
print({k: round(cmp[k].mean(), 2) for k in systems})
print(f"right in all 4: {(cmp.n_right == 4).sum()}, wrong in all 4: {(cmp.n_right == 0).sum()}")
retr = pd.read_csv("evaluation/results/results_retrieval.csv")
cmp.loc[cmp.n_right == 0, ["text", "category"]].assign(retrieval_output=retr.loc[cmp.n_right == 0, "output"])

{'zero_shot': np.float64(0.41), 'few_shot_constraint': np.float64(0.43), 'structured': np.float64(0.43), 'retrieval': np.float64(0.84)}
right in all 4: 24, wrong in all 4: 12


,text,category,retrieval_output
4,Why am I unable to take money from my card but the statement says the transa...,pending_cash_withdrawal,pending_transfer
9,What all currencies can be exchanged?,exchange_via_app,supported_cards_and_currencies
10,How do I exchange currency?,exchange_via_app,intent: supported_cards_and_currencies
29,On my last transaction it seem that my top-up was not successful.,top_up_reverted,top_up_failed
36,Why did I need to verfiy my identity?,why_verify_identity,verify_my_identity
49,Trouble transferring transaction contact for help,declined_transfer,intent not specified
63,"I am attempting to make a purchase online, but my card is not working. I kn...",reverted_card_payment?,transaction_charged_twice
66,What extra charges are there?,transfer_fee_charged,extra_charge_on_statement
67,Is it possible for me to get money out in a different currency?,receiving_money,supported_cards_and_currencies
68,What all is needed to verify the top-up card?,verify_top_up,verify_my_identity


## 9. E7 Final assistant (Req 10)

`main.py` combines the conclusions: retrieval prompt, greedy decoding for the intent, schema validation with the 77-name enum,
two tools with JSON schemas (`classify_intent`, `handle_intent`), and a sampled free-text reply.

The code fixes the tool order. We tested letting the 1.5B model drive the tool loop on 8 cases and it did not succeed once:
it skipped tools, invented an intent for `handle_intent`, or claimed actions it had not taken. With a small model, control belongs in code.

`handle_intent` is a simulation, as the assignment allows.

In [16]:
from main import handle, TOOLS, INTENT_SCHEMA

print("tool schemas:", [t["function"]["name"] for t in TOOLS])
for m in ["I lost my card, please block it",
          "Why is it asking me to verify my identity",
          "My top-up for Google Pay isn't working",
          "asdf qwerty zzz"]:
    print(f"\nCustomer:  {m}")
    print(f"Assistant: {handle(m)}")

tool schemas: ['classify_intent', 'handle_intent']

Customer:  I lost my card, please block it


16:28:07 llm: LLM call: 527 prompt tokens, decoding={'do_sample': False}


16:28:08 llm: LLM output (6 new tokens): 'lost_or_stolen_card'


16:28:08 baseline: parse_intent: found ['lost_or_stolen_card'] in output -> 'lost_or_stolen_card'


16:28:08 llm: LLM call: 397 prompt tokens, decoding={'do_sample': True, 'temperature': 0.7}


...

16:28:17 llm: LLM call: 591 prompt tokens, decoding={'do_sample': False}


Assistant: Your Apple Pay or Google Pay transaction has been completed successfully. Your reference number is REF-3597. If you have any further issues, please contact our support team.

Customer:  asdf qwerty zzz


16:28:18 llm: LLM output (13 new tokens): 'None of the provided messages match any of the given intents.'


16:28:18 baseline: parse_intent: found nothing in output -> ''


  [tool] classify_intent({'message': 'asdf qwerty zzz'}) -> {'error': "Could not map the message to a known intent (model said 'None of the provided messages match any of the given intents.')."}
Assistant: Sorry, I could not understand your request. Could you say it in a different way?


## 10. Summary

| Step | Result |
|---|---|
| E1 baseline | 0.41 |
| E2 best instruction prompt | 0.43; format fixed by the constraint line |
| E3 decoding | greedy is best for a one-label answer |
| E4 structured output | 100/100 JSON, 86/100 schema-valid; enum catches invented labels |
| Retrieval few-shot | **0.84**; the adaptation decision is RAG |
| Final system | retrieval + greedy + schema + code-driven tools + sampled reply |

Limits: 1.5B model on an 8 GB laptop; `handle_intent` is a stub; TF-IDF retrieval matches words, not meaning (see `docs/FAILURES.md`, type F).